# 02 — Preprocessing

**Farms:** A · B · C &nbsp;|&nbsp; **Input:** raw SCADA CSVs &nbsp;|&nbsp; **Output:** per-farm and combined `_avg_only.csv` files.

---

| Step | Description |
|------|-------------|
| 1 | Configuration & helpers |
| 2 | Load event metadata |
| 3 | Pipeline preview (single event) |
| 4 | Process all events for all farms |
| 5 | Validate & save |


## 1. Configuration & Helpers


In [2]:
# 02 — Preprocessing (Corrected for Time-Series + Streaming ML)

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT  = Path("..")
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

# -----------------------------
# 1. CONFIGURATION
# -----------------------------

FARMS = {
    "A": {
        "dataset_dir": RAW_DIR / "windfarmA" / "dataset",
        "event_info_path": RAW_DIR / "windfarmA" / "event_info.csv",
    },
    "B": {
        "dataset_dir": RAW_DIR / "windfarmB" / "datasets",
        "event_info_path": RAW_DIR / "windfarmB" / "event_info.csv",
    },
   
}

for farm, paths in FARMS.items():
    print(f"Farm {farm}:",
          paths["dataset_dir"].exists(),
          paths["event_info_path"].exists())


# -----------------------------
# 2. LOAD METADATA
# -----------------------------

event_info_all = {}

for farm, paths in FARMS.items():
    df = pd.read_csv(paths["event_info_path"], sep=";")
    event_info_all[farm] = df

    print(f"Farm {farm}: {len(df)} events "
          f"(normal={sum(df['event_label']=='normal')}, "
          f"anomaly={sum(df['event_label']=='anomaly')})")


# -----------------------------
# 3. HELPERS
# -----------------------------

def load_event(event_id, dataset_dir):
    path = dataset_dir / f"{event_id}.csv"
    return pd.read_csv(path, sep=";")


def keep_avg_columns(df):
    avg_cols = [c for c in df.columns if c.endswith("_avg")]
    return df[["time_stamp"] + avg_cols]


def convert_timestamp(df):
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
    return df


def clean_missing(df):
    # IMPORTANT: only forward fill (no leakage)
    return df.ffill()


def sort_time(df):
    return df.sort_values("time_stamp").reset_index(drop=True)


# -----------------------------
# 4. STORE EVENTS (IMPORTANT FIX)
# -----------------------------
# We KEEP EVENTS SEPARATED (no flattening)

farm_events = {}  # farm -> list of event dataframes

for farm, paths in FARMS.items():
    event_info = event_info_all[farm]

    events_list = []
    skipped = []

    print(f"\nProcessing Farm {farm}...")

    for _, row in event_info.iterrows():
        event_id = row["event_id"]
        label    = row["event_label"]

        file_path = paths["dataset_dir"] / f"{event_id}.csv"
        if not file_path.exists():
            skipped.append(event_id)
            continue

        df = load_event(event_id, paths["dataset_dir"])

        # pipeline
        df = keep_avg_columns(df)
        df = convert_timestamp(df)
        df = sort_time(df)
        df = clean_missing(df)

        # metadata
        df["event_id"] = event_id
        df["label"]    = label
        df["farm"]     = farm

        events_list.append(df)

    farm_events[farm] = events_list

    print(f"Farm {farm}: {len(events_list)} events loaded, skipped={len(skipped)}")


# -----------------------------
# 5. SLIDING WINDOW CREATION
# -----------------------------
# THIS IS CRITICAL FOR MODELING + STREAMING

def create_windows(df, window_size=50, step=5):
    feature_cols = [
        c for c in df.columns
        if c not in ["time_stamp", "event_id", "label", "farm"]
    ]

    X, y, meta = [], [], []

    for i in range(0, len(df) - window_size, step):
        window = df.iloc[i:i+window_size]

        X.append(window[feature_cols].values)

        # label = event-level label (later you can improve to horizon labeling)
        y.append(window["label"].iloc[0])

        meta.append({
            "event_id": window["event_id"].iloc[0],
            "farm": window["farm"].iloc[0],
            "start_time": window["time_stamp"].iloc[0],
            "end_time": window["time_stamp"].iloc[-1]
        })

    return np.array(X), np.array(y), meta


# -----------------------------
# 6. BUILD TRAINING SET (WINDOWED)
# -----------------------------

X_all, y_all, meta_all = [], [], []

for farm, events in farm_events.items():
    print(f"\nCreating windows for Farm {farm}...")

    for df in events:
        X, y, meta = create_windows(df)

        if len(X) == 0:
            continue

        X_all.append(X)
        y_all.append(y)
        meta_all.extend(meta)


X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print("\nFINAL SHAPES")
print("X:", X_all.shape)
print("y:", y_all.shape)


# -----------------------------
# 7. SAVE PROCESSED DATA
# -----------------------------

np.save(PROCESSED_DIR / "X_windows.npy", X_all)
np.save(PROCESSED_DIR / "y_labels.npy", y_all)

meta_df = pd.DataFrame(meta_all)
meta_df.to_parquet(PROCESSED_DIR / "window_metadata.parquet", index=False)

print("\nSaved:")
print("- X_windows.npy")
print("- y_labels.npy")
print("- window_metadata.parquet")

Farm A: True True
Farm B: True True
Farm A: 22 events (normal=11, anomaly=11)
Farm B: 15 events (normal=9, anomaly=6)

Processing Farm A...
Farm A: 22 events loaded, skipped=0

Processing Farm B...
Farm B: 15 events loaded, skipped=0

Creating windows for Farm A...

Creating windows for Farm B...


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 2, the array at index 0 has size 46 and the array at index 22 has size 63